# RealCause Lalonde HPO Benchmark: Foundation Models vs. Metalearners

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/layer6ai-labs/causalfm-survey/blob/main/notebooks/RealCause_with_hpo_benchmark.ipynb)

This production notebook uses the **CausalPFN arXiv v1 first-10 protocol** for RealCause draws from CPS and PSID. Current arXiv v2 uses all 100 and reports different values. Every CSV is an independent generator draw of covariates, treatment, and potential outcomes; X and T are **not fixed** across realizations.

V1 references: CPS PEHE **8.83 ± 0.04 (×10³)** and ATE relative error **0.08 ± 0.02**; PSID PEHE **13.98 ± 0.43 (×10³)** and ATE relative error **0.20 ± 0.03**. Saved PEHE remains in raw dollars; only the display divides PEHE/SEM by 1,000.

The run is pinned, seeded with deterministic algorithms requested (warn-only), CUDA-strict by default, foundation-first, fail-fast, atomically checkpointed, and resumable.

## 1. Exact environment before project imports

Run top-to-bottom in a fresh kernel. Colab installs the tested pins before importing torch or causal_bench. Locally, mismatches fail with an exact `uv pip install` command; no silent `%pip` fallback is attempted.

In [ ]:
import importlib.metadata as md
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
PINS = {
    "torch":"2.9.1", "numpy":"2.2.6", "pandas":"2.3.3",
    "scikit-learn":"1.6.1", "econml":"0.17.0", "FLAML":"2.3.5",
    "causalpfn":"0.1.4", "networkx":"3.4.2", "tqdm":"4.68.3",
    "einops":"0.8.2", "tabpfn":"2.0.9", "tensorboard":"2.21.0",
    "scipy":"1.15.3", "xgboost":"2.1.4", "lightgbm":"4.7.0",
}
PIP_SPECS = [
    "torch==2.9.1", "numpy==2.2.6", "pandas==2.3.3",
    "scikit-learn==1.6.1", "econml==0.17.0", "FLAML[automl]==2.3.5",
    "causalpfn==0.1.4", "networkx==3.4.2", "tqdm==4.68.3",
    "einops==0.8.2", "tabpfn==2.0.9", "tensorboard==2.21.0",
    "scipy==1.15.3", "xgboost==2.1.4", "lightgbm==4.7.0",
]
MODULES = {
    "torch":"torch", "numpy":"numpy", "pandas":"pandas",
    "scikit-learn":"sklearn", "econml":"econml", "FLAML":"flaml",
    "causalpfn":"causalpfn", "networkx":"networkx", "tqdm":"tqdm",
    "einops":"einops", "tabpfn":"tabpfn", "tensorboard":"tensorboard",
    "scipy":"scipy", "xgboost":"xgboost", "lightgbm":"lightgbm",
}
def version(dist):
    try: return md.version(dist)
    except md.PackageNotFoundError: return None
if "causal_bench" in sys.modules:
    raise RuntimeError("Restart: causal_bench was imported before environment verification.")
mismatch = {d:(version(d),v) for d,v in PINS.items() if version(d) != v}
if IN_COLAB and mismatch:
    loaded = [MODULES[d] for d in mismatch if MODULES[d] in sys.modules]
    subprocess.run([sys.executable,"-m","pip","install","--quiet",*PIP_SPECS], check=True)
    if loaded:
        raise RuntimeError(
            f"Exact pins were installed, but old modules are already live: {loaded}. "
            "Restart the Colab runtime now, then rerun from the top."
        )
elif mismatch:
    command = " ".join(repr(x) for x in PIP_SPECS)
    raise RuntimeError(
        f"Exact tested pins are required; found {mismatch}.\n"
        f"Run: uv pip install {command}\nThen restart the kernel."
    )
remaining = {d:(version(d),v) for d,v in PINS.items() if version(d) != v}
if remaining: raise RuntimeError(f"Pin verification failed: {remaining}")
print("All dependency pins verified before torch/causal_bench import.")

## 2. Project, vendor pins, CUDA, and model selection

Vendor SHAs are Do-PFN `90d67433...` and CausalFM `bb74ef72...`; model artifacts are checked before imports. Production defaults are N=10, HPO budget=900 seconds/search, CUDA required, all models, and fail-fast. Smoke/staged env overrides:

- `CAUSAL_BENCH_N_REALIZATIONS=1`, `CAUSAL_BENCH_HPO_TIME_BUDGET=1`
- `CAUSAL_BENCH_MODEL_FILTER=IPW`
- GPU foundation pass: `CAUSAL_BENCH_RUN_HPO=0`
- CPU HPO pass: `CAUSAL_BENCH_RUN_FOUNDATION=0`, `CAUSAL_BENCH_REQUIRE_CUDA=0`
- `CAUSAL_BENCH_OUTPUT_DIR=/persistent/path`
- Exact-upstream CausalPFN neighbor parity: `CAUSAL_BENCH_CAUSALPFN_CAP_NUM_NEIGHBOURS=0`
- CausalFM OOM fallback: set `CAUSAL_BENCH_CAUSALFM_QUERY_BATCH_SIZE=1024`, then lower if needed

With IPW CATE omitted, the full default launches **42 FLAML searches/realization, 420 total, 105 budget-hours**. HPO is CPU-side, so a costly GPU will mostly idle during it.

In [ ]:
if "causal_bench" in sys.modules:
    raise RuntimeError("Restart: causal_bench is already loaded from a potentially stale checkout.")
def run(command):
    p = subprocess.run(command, capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(f"Failed: {' '.join(command)}\n{p.stdout}\n{p.stderr}")
    return p.stdout.strip()
def local_root():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p/"causal_bench").is_dir() and (p/"notebooks").is_dir(): return p.resolve()
if IN_COLAB:
    PROJECT_ROOT = Path("/content/causalfm-survey")
    if PROJECT_ROOT.exists(): run(["git","-C",str(PROJECT_ROOT),"pull","--ff-only"])
    else: run(["git","clone","https://github.com/layer6ai-labs/causalfm-survey.git",str(PROJECT_ROOT)])
else:
    PROJECT_ROOT = local_root()
    if PROJECT_ROOT is None: raise RuntimeError("Run Jupyter from this repository.")
VENDORS = {
 "Do-PFN":("https://github.com/jr2021/Do-PFN.git","90d67433b43c4d52d752dc336070f525ff856e0b"),
 "CausalFM-toolkit":("https://github.com/yccm/CausalFM-toolkit.git","bb74ef729c70d274fe2bb422c3b0edff4754fa37"),
}
def pin(name, url, sha):
    path = PROJECT_ROOT/"notebooks"/name
    if not path.exists(): run(["git","clone",url,str(path)])
    head = run(["git","-C",str(path),"rev-parse","HEAD"])
    if head != sha:
        dirty = run(["git","-C",str(path),"status","--porcelain","--untracked-files=no"])
        if dirty: raise RuntimeError(f"Refusing to overwrite dirty {name}:\n{dirty}")
        run(["git","-C",str(path),"fetch","origin",sha])
        run(["git","-C",str(path),"checkout","--detach",sha])
    got = run(["git","-C",str(path),"rev-parse","HEAD"])
    if got != sha: raise RuntimeError(f"{name}: {got} != {sha}")
    return path.resolve(), got
DOPFN_DIR,dopfn_sha = pin("Do-PFN",*VENDORS["Do-PFN"])
CAUSALFM_DIR,causalfm_sha = pin("CausalFM-toolkit",*VENDORS["CausalFM-toolkit"])
VENDOR_SHAS = {"Do-PFN":dopfn_sha,"CausalFM-toolkit":causalfm_sha}
CAUSALFM_CHECKPOINT = CAUSALFM_DIR/"checkpoints/checkpoints_standard/best_model.pth"
ARTIFACTS = [
 (CAUSALFM_CHECKPOINT,10_000_000),
 (DOPFN_DIR/"artifacts/model_submitit_0ccc_id_171b69db_epoch_-1.cpkt",10_000_000),
 (DOPFN_DIR/"artifacts/dopfn_model.pkl",10_000_000),
 (DOPFN_DIR/"artifacts/dopfn_config.pkl",100),
]
for path,minimum in ARTIFACTS:
    if not path.is_file() or path.stat().st_size < minimum:
        raise RuntimeError(f"Missing/truncated artifact: {path}")
    with path.open("rb") as f: header=f.read(128)
    if header.startswith(b"version https://git-lfs"):
        raise RuntimeError(f"Git-LFS pointer instead of artifact: {path}")
for p in reversed((PROJECT_ROOT,DOPFN_DIR,CAUSALFM_DIR)):
    if str(p) not in sys.path: sys.path.insert(0,str(p))

import gc, hashlib, json, random, re, time, traceback, warnings
from datetime import datetime, timezone
import numpy as np, pandas as pd, torch
def flag(name, default):
    x=os.environ.get(name,"1" if default else "0").strip().lower()
    if x in {"1","true","yes","on"}: return True
    if x in {"0","false","no","off"}: return False
    raise ValueError(f"Bad boolean {name}={x!r}")
REQUIRE_CUDA=flag("CAUSAL_BENCH_REQUIRE_CUDA",True)
FAIL_FAST=flag("CAUSAL_BENCH_FAIL_FAST",True)
CAUSALPFN_CAP_NEIGHBOURS=flag("CAUSAL_BENCH_CAUSALPFN_CAP_NUM_NEIGHBOURS",True)
_causalfm_batch=int(os.environ.get("CAUSAL_BENCH_CAUSALFM_QUERY_BATCH_SIZE","0"))
if _causalfm_batch < 0:
    raise ValueError("CAUSAL_BENCH_CAUSALFM_QUERY_BATCH_SIZE must be 0 or positive")
CAUSALFM_QUERY_BATCH_SIZE=None if _causalfm_batch==0 else _causalfm_batch
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("CUDA required. Select GPU, or set CAUSAL_BENCH_REQUIRE_CUDA=0 for HPO-only CPU.")
device="cuda" if torch.cuda.is_available() else "cpu"
BASE_SEED=82718
random.seed(BASE_SEED); np.random.seed(BASE_SEED); torch.manual_seed(BASE_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(BASE_SEED)
torch.use_deterministic_algorithms(True,warn_only=True)
torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
warnings.filterwarnings("default")

from causal_bench import (
 CausalPFNWrapper,DoPFNWrapper,CausalFMWrapper,HPOConfig,
 SLearnerWrapper,TLearnerWrapper,XLearnerWrapper,DebiasedMLWrapper,IPWWrapper,DRWrapper,
)
budget=float(os.environ.get("CAUSAL_BENCH_HPO_TIME_BUDGET","900"))
if not np.isfinite(budget) or budget<=0: raise ValueError("HPO budget must be positive.")
HPO_CONFIG=HPOConfig(time_budget=budget,cv=3,verbose=0,early_stop=True)
MODELS=[
 ("CausalPFN (Foundation)","CausalPFN",CausalPFNWrapper,"foundation"),
 ("Do-PFN (Foundation)","Do-PFN",DoPFNWrapper,"foundation"),
 ("CausalFM (Foundation)","CausalFM",CausalFMWrapper,"foundation"),
 ("S-learner","S-learner",SLearnerWrapper,"hpo"),
 ("T-learner","T-learner",TLearnerWrapper,"hpo"),
 ("X-learner","X-learner",XLearnerWrapper,"hpo"),
 ("Debiased ML","Debiased ML",DebiasedMLWrapper,"hpo"),
 ("IPW","IPW",IPWWrapper,"hpo"),
 ("DR (Doubly Robust)","DR (Doubly Robust)",DRWrapper,"hpo"),
]
SPECS=[{"display":d,"base":b,"cls":c,"kind":k,"supports_cate":bool(getattr(c,"supports_cate",True))} for d,b,c,k in MODELS]
run_f=flag("CAUSAL_BENCH_RUN_FOUNDATION",True); run_h=flag("CAUSAL_BENCH_RUN_HPO",True)
tokens={x.strip().casefold() for x in os.environ.get("CAUSAL_BENCH_MODEL_FILTER","").split(",") if x.strip()}
known={x.casefold() for s in SPECS for x in (s["display"],s["base"])}
if tokens-known: raise ValueError(f"Unknown model filter: {sorted(tokens-known)}")
SELECTED=[s for s in SPECS if ((s["kind"]=="foundation" and run_f) or (s["kind"]=="hpo" and run_h)) and (not tokens or s["display"].casefold() in tokens or s["base"].casefold() in tokens)]
if not SELECTED: raise RuntimeError("No models selected.")
def available(s):
    if s["base"]=="Do-PFN": return s["cls"].is_available(repo_dir=str(DOPFN_DIR))
    if s["base"]=="CausalFM": return s["cls"].is_available(checkpoint_path=str(CAUSALFM_CHECKPOINT))
    return s["cls"].is_available()
missing=[s["display"] for s in SELECTED if not available(s)]
if missing: raise RuntimeError(f"Selected models unavailable (never silently skipped): {missing}")
print("Device:",device,"FAIL_FAST:",FAIL_FAST,"HPO:",HPO_CONFIG)
print("CausalPFN cap_num_neighbours:",CAUSALPFN_CAP_NEIGHBOURS,
      "CausalFM query_batch_size:",CAUSALFM_QUERY_BATCH_SIZE)
print("Selected foundation-first:",[s["display"] for s in SELECTED])
if any(s["kind"]=="hpo" for s in SELECTED):
    print("WARNING: HPO is CPU-side; production default is 420 searches / 105 budget-hours.")

## 3. V1 data protocol and resumable execution

Default N is exactly 10. Each independent CSV uses `default_rng(42 + i)`, a 90/10 train/test split, PEHE on held-out true ITE, and a **fresh full-data fit** for ATE against `mean(ite)`. CPS runs first as the large-cohort stress test. IPW has `supports_cate=False`, so no meaningless CATE fit or PEHE is produced.

The wrapper named Debiased ML is generic EconML DML, not the paper's Forest DML, so its row is not a direct reproduction of that paper baseline. CausalPFN's default safe neighbor cap avoids FAISS `-1` sentinels and produced about a 0.1% deviation from exact upstream behavior in the audited sample; set the documented parity switch to 0 to disable it. CausalFM defaults to native unbatched inference because every batch recomputes the full context; if CPS OOMs, explicitly try 1024 and lower it further if necessary.

Inputs are downloaded from immutable CausalPFN revision `7fae8e26e4e584c99723aaf719f3b9627b369de7`; SHA-256 for every selected first-N cached CSV is recorded in the schema-3 manifest before resume is allowed.

Every successful model/cohort/realization is an atomic JSON checkpoint. Failure tracebacks are separate atomic JSON records. Default `FAIL_FAST=True` records and cleans up, then re-raises the first error.

In [ ]:
from causal_bench import load_lalonde_realcause,evaluate_cate,ate_abs_error,ate_rel_error
import causal_bench.data_loader as realcause_loader
N=int(os.environ.get("CAUSAL_BENCH_N_REALIZATIONS","10"))
if not 1<=N<=100: raise ValueError("CAUSAL_BENCH_N_REALIZATIONS must be 1..100")
COHORTS=("cps","psid"); SPLIT_SEED=42
batch_label="native" if CAUSALFM_QUERY_BATCH_SIZE is None else str(CAUSALFM_QUERY_BATCH_SIZE)
stem=(f"realcause-lalonde-first{N}_hpo{budget:g}s_cv3_split42_seed{BASE_SEED}"
      f"_cpfncap{int(CAUSALPFN_CAP_NEIGHBOURS)}_cfmbatch{batch_label}")
out=Path(os.environ.get("CAUSAL_BENCH_OUTPUT_DIR",str(PROJECT_ROOT/"notebooks/results"))).expanduser().resolve()/stem
checkpoints=out/"checkpoints"; failures=out/"failures"; csv_path=out/f"{stem}.csv"; manifest_path=out/"manifest.json"
for p in (out,checkpoints,failures): p.mkdir(parents=True,exist_ok=True)
def atomic_json(path,obj):
    tmp=path.with_name(f".{path.name}.{os.getpid()}.tmp")
    with tmp.open("w",encoding="utf-8") as f:
        json.dump(obj,f,indent=2,sort_keys=True,allow_nan=False); f.write("\n")
    os.replace(tmp,path)
def digest(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""): h.update(chunk)
    return h.hexdigest()
sources=[PROJECT_ROOT/"causal_bench"/x for x in ("wrap_foundation.py","wrap_metalearners.py","wrap_causalpfn.py","wrap_dopfn.py","wrap_causalfm.py","data_loader.py","metrics.py")]
REALCAUSE_DATA_REVISION="7fae8e26e4e584c99723aaf719f3b9627b369de7"
if realcause_loader._REALCAUSE_REVISION != REALCAUSE_DATA_REVISION:
    raise RuntimeError(
        "Loader/data revision mismatch: "
        f"{realcause_loader._REALCAUSE_REVISION} != {REALCAUSE_DATA_REVISION}"
    )
data_sha256={}
data={}
for cohort in COHORTS:
    rs=load_lalonde_realcause(cohort,n_realizations=N,seed=SPLIT_SEED,test_ratio=.1)
    if len(rs)!=N: raise RuntimeError(f"{cohort}: expected {N}, got {len(rs)}")
    for i,r in enumerate(rs):
        arrays=(r.X_train,r.T_train,r.Y_train,r.X_test,r.tau_true_test,r.X_full,r.T_full,r.Y_full)
        if r.realization!=i or not all(np.isfinite(np.asarray(a)).all() for a in arrays):
            raise RuntimeError(f"{cohort} realization {i}: invalid data")
        if not np.isfinite(r.ate_true):
            raise RuntimeError(f"{cohort} realization {i}: non-finite ate_true")
        if any(np.asarray(x).ndim != 2 for x in (r.X_train,r.X_test,r.X_full)):
            raise RuntimeError(f"{cohort} realization {i}: X must be 2D")
        if not (len(r.X_train)==len(r.T_train)==len(r.Y_train)):
            raise RuntimeError(f"{cohort} realization {i}: train length mismatch")
        if not (len(r.X_test)==len(r.tau_true_test)):
            raise RuntimeError(f"{cohort} realization {i}: test length mismatch")
        if not (len(r.X_full)==len(r.T_full)==len(r.Y_full)):
            raise RuntimeError(f"{cohort} realization {i}: full length mismatch")
        if not (r.X_train.shape[1]==r.X_test.shape[1]==r.X_full.shape[1]):
            raise RuntimeError(f"{cohort} realization {i}: feature-count mismatch")
        if set(np.unique(r.T_train))!={0.,1.} or set(np.unique(r.T_full))!={0.,1.}:
            raise RuntimeError(f"{cohort} realization {i}: invalid treatment")
    for i in range(N):
        cache_path=Path(realcause_loader._CACHE_DIR)/f"realcause_lalonde_{cohort}_sample{i}.csv"
        if not cache_path.is_file():
            raise RuntimeError(f"Expected cached RealCause input is missing: {cache_path}")
        data_sha256[cache_path.name]=digest(cache_path)
    data[cohort]=rs
    print(cohort,len(rs),rs[0].meta)
if len(data_sha256) != len(COHORTS)*N:
    raise RuntimeError(
        f"Expected {len(COHORTS)*N} RealCause hashes, got {len(data_sha256)}"
    )
config={
 "schema":3,"n":N,"cohorts":list(COHORTS),"split_seed":SPLIT_SEED,"base_seed":BASE_SEED,
 "hpo":{"time_budget":budget,"cv":3},"models":[s["display"] for s in SPECS],
 "foundation":{"causalpfn_cap_num_neighbours":CAUSALPFN_CAP_NEIGHBOURS,
               "causalfm_query_batch_size":CAUSALFM_QUERY_BATCH_SIZE},
 "versions":{d:md.version(d) for d in PINS},"vendor_shas":VENDOR_SHAS,
 "source_sha256":{str(p.relative_to(PROJECT_ROOT)):digest(p) for p in sources},
 "realcause_data":{"revision":REALCAUSE_DATA_REVISION,
                   "csv_sha256":dict(sorted(data_sha256.items()))},
}
if manifest_path.exists():
    with manifest_path.open(encoding="utf-8") as f: old=json.load(f)
    if old.get("config")!=config: raise RuntimeError(f"Resume manifest mismatch: {manifest_path}")
else:
    atomic_json(
        manifest_path,
        {"created_utc":datetime.now(timezone.utc).isoformat(),"config":config},
    )
print("RealCause input revision:",REALCAUSE_DATA_REVISION)
print("Hashed cached RealCause CSVs:",len(data_sha256))
print("Output:",out)

In [ ]:
def slug(x): return re.sub(r"[^a-z0-9]+","-",x.casefold()).strip("-")
def key(model,cohort,i): return model,cohort,int(i)
def task_path(k): return checkpoints/f"{slug(k[0])}__{k[1]}__r{k[2]:02d}.json"
def task_seed(mi,ci,i,stage): return BASE_SEED+mi*100_000+ci*10_000+int(i)*10+(1 if stage=="cate" else 2)
def seed_all(x):
    random.seed(x); np.random.seed(x); torch.manual_seed(x)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(x)
def cleanup():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
def params(model):
    return json.dumps(getattr(model,"best_params_",{}) or {},sort_keys=True,default=repr)
def vector(x,n,label):
    x=np.asarray(x,dtype=float).reshape(-1)
    if len(x)!=n or not np.isfinite(x).all(): raise RuntimeError(f"Invalid {label}: shape={x.shape}")
    return x
def make(s):
    if s["base"]=="CausalPFN":
        return s["cls"](device=device,cap_num_neighbours=CAUSALPFN_CAP_NEIGHBOURS)
    if s["base"]=="CausalFM":
        return s["cls"](checkpoint_path=str(CAUSALFM_CHECKPOINT),device=device,
                        query_batch_size=CAUSALFM_QUERY_BATCH_SIZE)
    if s["base"]=="Do-PFN": return s["cls"](repo_dir=str(DOPFN_DIR),device=device)
    if s["kind"]=="foundation": return s["cls"](device=device)
    return s["cls"](device="cpu",hpo=True,hpo_config=HPO_CONFIG)
all_keys={key(s["display"],c,i) for s in SPECS for c in COHORTS for i in range(N)}
selected_keys={key(s["display"],c,i) for s in SELECTED for c in COHORTS for i in range(N)}
def read_records():
    ans={}
    for p in sorted(checkpoints.glob("*.json")):
        with p.open(encoding="utf-8") as f: row=json.load(f)
        k=key(row["model"],row["cohort"],row["realization"])
        if k not in all_keys or k in ans: raise RuntimeError(f"Bad/duplicate checkpoint: {p}")
        ans[k]=row
    return ans
def rebuild(records):
    frame=pd.DataFrame(sorted(records.values(),key=lambda r:(r["model"],r["cohort"],r["realization"])))
    tmp=csv_path.with_name(f".{csv_path.name}.{os.getpid()}.tmp")
    frame.to_csv(tmp,index=False); os.replace(tmp,csv_path); return frame
def fail(s,c,i,stage,seed,exc):
    now=datetime.now(timezone.utc)
    atomic_json(failures/f"{now.strftime('%Y%m%dT%H%M%S.%fZ')}__{os.getpid()}__{slug(s['display'])}__{c}__r{i:02d}.json",{
      "timestamp_utc":now.isoformat(),"model":s["display"],"cohort":c,"realization":i,
      "stage":stage,"seed":seed,"exception_type":type(exc).__name__,"exception":str(exc),
      "traceback":traceback.format_exc(),
    })
records=read_records()
print(f"Resume: {len(records)}/{len(all_keys)} total; {len(set(records)&selected_keys)}/{len(selected_keys)} selected.")

In [ ]:
for mi,s in enumerate(SPECS):
    if s not in SELECTED: continue
    for ci,cohort in enumerate(COHORTS):
        for r in data[cohort]:
            k=key(s["display"],cohort,r.realization)
            if k in records: continue
            cate=ate=None; stage="init"; active_seed=None
            try:
                tau=lower=upper=None; cate_time=None; cate_params=None
                if s["supports_cate"]:
                    stage="cate"; active_seed=task_seed(mi,ci,r.realization,stage); seed_all(active_seed)
                    start=time.perf_counter(); cate=make(s); cate.fit(r.X_train,r.T_train,r.Y_train)
                    pred=cate.predict(r.X_test)
                    if not isinstance(pred,tuple) or len(pred)!=3: raise RuntimeError("predict must return 3-tuple")
                    tau,lower,upper=pred; tau=vector(tau,len(r.X_test),"CATE")
                    if (lower is None)!=(upper is None): raise RuntimeError("Incomplete interval")
                    if lower is not None:
                        lower=vector(lower,len(tau),"lower"); upper=vector(upper,len(tau),"upper")
                        if np.any(lower>upper): raise RuntimeError("lower > upper")
                    cate_time=time.perf_counter()-start; cate_params=params(cate)
                    cate=None; cleanup()
                stage="ate"; active_seed=task_seed(mi,ci,r.realization,stage); seed_all(active_seed)
                start=time.perf_counter(); ate=make(s); ate.fit(r.X_full,r.T_full,r.Y_full)
                if hasattr(ate,"estimate_ate"): ate_hat=ate.estimate_ate(r.X_full,r.T_full,r.Y_full)
                else:
                    full,_,_=ate.predict(r.X_full); ate_hat=float(vector(full,len(r.X_full),"full CATE").mean())
                ate_array=np.asarray(ate_hat)
                if ate_array.size != 1:
                    raise RuntimeError(f"ATE output must be scalar; got shape {ate_array.shape}")
                ate_hat=float(ate_array.reshape(-1)[0])
                if not np.isfinite(ate_hat): raise RuntimeError("Non-finite ATE")
                ate_time=time.perf_counter()-start; ate_params=params(ate)
                ate=None; cleanup()
                if s["supports_cate"]:
                    metrics=evaluate_cate(tau,r.tau_true_test,ate_hat=ate_hat,ate_true=r.ate_true,lower=lower,upper=upper,runtime_s=cate_time+ate_time)
                else:
                    metrics={"pehe":None,"ate_hat":ate_hat,"ate_true":float(r.ate_true),
                     "ate_abs_error":ate_abs_error(ate_hat,r.ate_true),"ate_rel_error":ate_rel_error(ate_hat,r.ate_true),
                     "bias":None,"coverage_95":None,"runtime_s":ate_time}
                row={**metrics,"model":s["display"],"base_model":s["base"],"model_kind":s["kind"],
                 "supports_cate":s["supports_cate"],"cohort":cohort,"realization":int(r.realization),
                 "execution_device":device if s["kind"]=="foundation" else "cpu",
                 "cate_runtime_s":cate_time,"ate_runtime_s":ate_time,"cate_best_params":cate_params,
                 "ate_best_params":ate_params,"cate_seed":task_seed(mi,ci,r.realization,"cate") if s["supports_cate"] else None,
                 "ate_seed":task_seed(mi,ci,r.realization,"ate"),"hpo_time_budget_s":budget if s["kind"]=="hpo" else None,
                 "hpo_cv":3 if s["kind"]=="hpo" else None,"split_seed":SPLIT_SEED,"base_seed":BASE_SEED}
                atomic_json(task_path(k),row); records[k]=row; rebuild(records)
                print(s["display"],cohort,r.realization,"OK")
            except Exception as exc:
                fail(s,cohort,r.realization,stage,active_seed,exc)
                print(s["display"],cohort,r.realization,"FAILED",stage,repr(exc))
                if FAIL_FAST: raise
            finally:
                cate=None; ate=None; cleanup()
    clear_cache=getattr(s["cls"],"clear_model_cache",None)
    if clear_cache is not None:
        removed=clear_cache()
        print(f"{s['display']} cleared {removed} cached model(s)")
    cleanup()
frame=rebuild(records)
print(f"Checkpointed {len(records)}/{len(all_keys)} total tasks.")

## 4. Results and completion

PEHE display is ×10³ for direct v1 comparison; raw CSV units are unchanged. Counts and completeness are explicit. Completeness is scoped to this invocation's selected models, enabling staged GPU foundation and CPU HPO jobs.

In [ ]:
records=read_records(); frame=rebuild(records); chosen={s["display"] for s in SELECTED}; rows=[]
for s in SPECS:
  for cohort in COHORTS:
    sub=frame[(frame["model"]==s["display"])&(frame["cohort"]==cohort)] if not frame.empty else pd.DataFrame()
    n=len(sub)
    if s["supports_cate"] and n:
        x=pd.to_numeric(sub["pehe"],errors="coerce").dropna()/1000
        pehe=f"{x.mean():.2f} ± {x.sem():.2f}" if len(x)>1 else f"{x.mean():.2f} ± N/A"
    elif s["supports_cate"]: pehe="pending"
    else: pehe="N/A"
    if n:
        x=pd.to_numeric(sub["ate_rel_error"],errors="coerce").dropna()
        ater=f"{x.mean():.2f} ± {x.sem():.2f}" if len(x)>1 else f"{x.mean():.2f} ± N/A"
    else: ater="pending"
    selected=s["display"] in chosen
    status=("complete" if n==N else "INCOMPLETE") if selected else "not selected"
    rows.append({"model":s["display"],"cohort":cohort.upper(),"PEHE (×10³), mean ± SEM":pehe,
      "ATE relative error, mean ± SEM":ater,"completed":f"{n}/{N}","status":status})
summary=pd.DataFrame(rows)
print("RealCause Lalonde",("arXiv v1 first-10" if N==10 else f"first-{N} override"))
print(summary.to_string(index=False))
print("\nRaw CSV:",csv_path,"\nCheckpoints:",checkpoints,"\nFailures:",failures,len(list(failures.glob("*.json"))),"\nManifest:",manifest_path)
missing=sorted(selected_keys-set(records))
if missing: raise RuntimeError(f"Incomplete after saving: {len(missing)}/{len(selected_keys)} selected tasks missing; first={missing[:10]}")
print(f"SUCCESS: all {len(selected_keys)} selected tasks complete.")